In [4]:
from google.cloud import bigquery
from google.cloud import storage
import numpy as np
import pandas as pd
import geopandas as gpd
import datetime as dt
import matplotlib.pyplot as plt
from decimal import *
from calendar import monthrange, month_abbr
import cartopy as ctp
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.io import shapereader
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import io
import pygrib
import math
import netCDF4
import xarray as xr
import warnings


In [5]:
def points_with_observations_and_historical_data():
    """
    gives the list of gridpoints (from the anagraphical forecast table) that have a univoque station associated, and for which we have the historic of the forecasts.
    In the dataset:
    ID -> ID of the gridpoint
    ID_CONSUNTIVO -> ID of the univoque station
    LAT_METEOMATICS, LON_METEOMATICS -> coordinates of the point given for which the data have been given by Meteomatics
    MIN_LAT_MOLOCH, MIN_LON_MOLOCH -> coordinates of the closest Moloch gridpoint 
    """

    # define the client for the queries
    project_name = 'a2a-dataplatformgt-pmt-prd'
    client = bigquery.Client(project=project_name)
    
    # query
    query = """
        SELECT ANA.*
        FROM `a2a-dataplatformgt-dwh-prd.L2.D_ANAGRAFICA_METEO_FCST` ANA
        WHERE ANA.LAT=(SELECT DISTINCT FCST.LAT, 
            FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
            WHERE ANA.LAT=FCST.LAT AND ANA.LON=FCST.LON AND FCST.TRADE_DATE < "2023-01-01T00:00:00"
            )
        AND ANA.LON=(SELECT DISTINCT FCST.LON, 
            FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
            WHERE ANA.LAT=FCST.LAT AND ANA.LON=FCST.LON AND FCST.TRADE_DATE < "2023-01-01T00:00:00"
            ) 
        AND ANA.ID_CONSUNTIVO=(SELECT DISTINCT OBS.ID_CONSUNTIVO, 
            FROM `a2a-dataplatformgt-dwh-prd.L2.D_ANAGRAFICA_METEO_OBS` OBS
            WHERE ANA.ID_CONSUNTIVO=OBS.ID_CONSUNTIVO AND OBS.STORICO='DS' 
            )

        ORDER BY ANA.ID;
        """

    result = client.query(query)

    # save the result in a pandas dataset
    dataset = result.to_dataframe(create_bqstorage_client=False).astype({'LAT':float, 'LON':float})
    dataset=dataset.set_index('ID')

    return dataset



In [9]:
gridpoints=points_with_observations_and_historical_data()
print(gridpoints)

        ID_PAST  ID_CONSUNTIVO                 ASSET_ID  \
ID                                                        
900249     <NA>         502484               AGGIUNTIVI   
900252     <NA>         502527               AGGIUNTIVI   
900253     <NA>         502431               AGGIUNTIVI   
900254     <NA>         502222               AGGIUNTIVI   
900288     <NA>         502094         FV_BECHI_MACOMER   
...         ...            ...                      ...   
904626     <NA>         501253  IBM_Weather_Underground   
904648     <NA>         503449  IBM_Weather_Underground   
904668     <NA>         501033  IBM_Weather_Underground   
904672     <NA>         501162  IBM_Weather_Underground   
904685     <NA>         501288  IBM_Weather_Underground   

                              ASSET_ID_WTG  \
ID                                           
900249            AGGIUNTIVI_14_WN_MIMIANI   
900252            AGGIUNTIVI_17_WN_MIMIANI   
900253          AGGIUNTIVI_18_WN_Matarocco   
90

In [11]:
def query_ecop_ponctual(list_coordinates, var, list_forecast_date):
    """
    get the timeseries of the ecop forecast for the list of points 
    """

    project_name = 'a2a-dataplatformgt-pmt-prd'
    client = bigquery.Client(project=project_name)

    query = """
        SELECT FCST.*
        FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
        WHERE FCST.LAT IN UNNEST(@list_lat)
            AND FCST.LON IN UNNEST(@list_lon)
            AND FCST.COD_DATA_TYPE= @var
            AND FCST.TRADE_DATE in UNNEST(@forecast_dates)
        ORDER BY FCST.VALID_DATE;
        """
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("list_lat", "FLOAT", [lat for lat,_ in list_coordinates]),
            bigquery.ArrayQueryParameter("list_lon", "FLOAT", [lon for _,lon in list_coordinates]),
            bigquery.ArrayQueryParameter("forecast_dates", "DATETIME", list_forecast_date),
            bigquery.ScalarQueryParameter("var", "STRING", var),
            ]
        )
    
    result = client.query(query, job_config=job_config)
    dataset = result.to_dataframe(create_bqstorage_client=False)
    dataset = dataset.astype({'LAT':float, 'LON':float, 'VALUE':float})

    return dataset

In [12]:
def query_all_forecasts(gridpoints):
    """
    Get all forecasts for the gridpoints provided.
    """

    project_name = 'a2a-dataplatformgt-pmt-prd'
    client = bigquery.Client(project=project_name)

    query = """
        SELECT FCST.*
        FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
        WHERE FCST.LAT IN UNNEST(@list_lat)
            AND FCST.LON IN UNNEST(@list_lon)
        ORDER BY FCST.VALID_DATE;
        """
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("list_lat", "FLOAT", gridpoints['LAT'].tolist()),
            bigquery.ArrayQueryParameter("list_lon", "FLOAT", gridpoints['LON'].tolist())
        ]
    )
    
    result = client.query(query, job_config=job_config)
    dataset = result.to_dataframe(create_bqstorage_client=False)
    dataset = dataset.astype({'LAT': float, 'LON': float, 'VALUE': float})

    return dataset

In [ ]:
def query_and_save_forecasts(gridpoints, output_file, var, year, month):
    """
    Query forecasts for each trade date for the specified year and month, and save the results in a netCDF file.
    """
    project_name = 'a2a-dataplatformgt-pmt-prd'
    client = bigquery.Client(project=project_name)

    # Query distinct trade dates
    trade_dates_query = """
        SELECT DISTINCT FCST.TRADE_DATE
        FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
        WHERE FCST.LAT IN UNNEST(@list_lat)
            AND FCST.LON IN UNNEST(@list_lon)
            AND FCST.COD_DATA_TYPE = @var
            AND FCST.TRADE_DATE >= @start_date
            AND EXTRACT(MONTH FROM FCST.TRADE_DATE) = @month
            AND EXTRACT(YEAR FROM FCST.TRADE_DATE) = @year
        ORDER BY FCST.TRADE_DATE;
    """
    
    trade_dates_job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("list_lat", "FLOAT", gridpoints['LAT'].tolist()),
            bigquery.ArrayQueryParameter("list_lon", "FLOAT", gridpoints['LON'].tolist()),
            bigquery.ScalarQueryParameter("var", "STRING", var),
            bigquery.ScalarQueryParameter("start_date", "DATETIME", f"{year}-{month}-01T00:00:00"),
            bigquery.ScalarQueryParameter("month", "INT64", month),
            bigquery.ScalarQueryParameter("year", "INT64", year)
        ]
    )
    
    trade_dates_result = client.query(trade_dates_query, job_config=trade_dates_job_config)
    trade_dates = [row.TRADE_DATE for row in trade_dates_result]

    # Initialize an empty xarray Dataset
    ds = xr.Dataset()

    # Perform a single query to get all forecasts for the specified trade dates
    query = """
        SELECT FCST.*
        FROM `a2a-dataplatformgt-dwh-prd.L2.F_FORECAST_DATA_ECMWF_IFS_METEOMATICS` FCST
        WHERE FCST.LAT IN UNNEST(@list_lat)
            AND FCST.LON IN UNNEST(@list_lon)
        AND FCST.TRADE_DATE IN UNNEST(@trade_dates)
            AND FCST.COD_DATA_TYPE = @var
        ORDER BY FCST.VALID_DATE;
        """
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("list_lat", "FLOAT", gridpoints['LAT'].tolist()),
            bigquery.ArrayQueryParameter("list_lon", "FLOAT", gridpoints['LON'].tolist()),
            bigquery.ArrayQueryParameter("trade_dates", "DATETIME", trade_dates[:6]),
            bigquery.ScalarQueryParameter("var", "STRING", var)
        ]
    )
    
    result = client.query(query, job_config=job_config)
    dataset = result.to_dataframe(create_bqstorage_client=False)
    dataset = dataset.astype({'LAT': float, 'LON': float, 'VALUE': float})

    # Convert the datetime values to nanosecond precision
    dataset['VALID_DATE'] = dataset['VALID_DATE'].astype('datetime64[ns]')

    print(dataset)

    # Ensure all columns have data types that xarray can handle
    dataset = dataset.astype({
        'LAT': 'float64',
        'LON': 'float64',
        'TRADE_DATE': 'datetime64[ns]',
        'VALID_DATE': 'datetime64[ns]',
        'VALUE': 'float64',
        'FLOW_DATE': 'datetime64[ns]'
    })

    # Convert the pandas DataFrame to an xarray Dataset
    ds = dataset.set_index(['LAT', 'LON', 'TRADE_DATE', 'VALID_DATE']).to_xarray()

    # Save the dataset to a netCDF file
    ds.to_netcdf(output_file)


In [21]:
var='T_2M_C'
year=2023
with warnings.catch_warnings(action="ignore"):
    for month in range(1, 13):
        output_file = 'forecasts_{}{}_{}.nc'.format(year, month, var)
        query_and_save_forecasts(gridpoints, output_file, var, year, month)

[datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 3, 0, 0), datetime.datetime(2023, 1, 4, 0, 0), datetime.datetime(2023, 1, 5, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 7, 0, 0), datetime.datetime(2023, 1, 8, 0, 0), datetime.datetime(2023, 1, 9, 0, 0), datetime.datetime(2023, 1, 10, 0, 0), datetime.datetime(2023, 1, 11, 0, 0), datetime.datetime(2023, 1, 12, 0, 0), datetime.datetime(2023, 1, 13, 0, 0), datetime.datetime(2023, 1, 14, 0, 0), datetime.datetime(2023, 1, 15, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 17, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.datetime(2023, 1, 19, 0, 0), datetime.datetime(2023, 1, 20, 0, 0), datetime.datetime(2023, 1, 21, 0, 0), datetime.datetime(2023, 1, 22, 0, 0), datetime.datetime(2023, 1, 23, 0, 0), datetime.datetime(2023, 1, 24, 0, 0), datetime.datetime(2023, 1, 25, 0, 0), datetime.datetime(2023, 1, 26, 0, 0), datetime.datetime(20

KeyboardInterrupt: 

In [ ]:
# DOES NOT WORK

def query_and_save_observations(gridpoints, output_file, var, year, month):
    """
    Query observations for each date for the specified year and month, and save the results in a netCDF file.
    """
    project_name = 'a2a-dataplatformgt-pmt-prd'
    client = bigquery.Client(project=project_name)

    # Initialize an empty xarray Dataset
    ds = xr.Dataset()

    # Perform a single query to get all observations for the specified dates
    query = """
        SELECT OBS.*
        FROM `a2a-dataplatformgt-dwh-prd.L2.F_OBSERVATION_DATA_MISTRAL` OBS
        WHERE OBS.LATITUDE IN UNNEST(@list_lat)
            AND OBS.LONGITUDE IN UNNEST(@list_lon)
            AND OBS.STATION_NAME IN UNNEST(@list_station_name_)
            AND EXTRACT(MONTH FROM OBS.REF_TIME) = @month
            AND EXTRACT(YEAR FROM OBS.REF_TIME) = @year
            AND OBS.PRODUCT = @var
        ORDER BY OBS.REF_TIME;
    """
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("list_lat", "FLOAT", gridpoints['LAT'].tolist()),
            bigquery.ArrayQueryParameter("list_lon", "FLOAT", gridpoints['LON'].tolist()),
            bigquery.ScalarQueryParameter("var", "STRING", var),
            bigquery.ArrayQueryParameter("list_station_name_", "STRING", list_station_name),
            bigquery.ScalarQueryParameter("month", "INT64", month),
            bigquery.ScalarQueryParameter("year", "INT64", year)
        ]
    )
    
    result = client.query(query, job_config=job_config)
    dataset = result.to_dataframe(create_bqstorage_client=False)
    dataset = dataset.astype({'LATITUDE': float, 'LONGITUDE': float, 'VALUE': float})

    # Convert the datetime values to nanosecond precision
    dataset['REF_TIME'] = dataset['REF_TIME'].astype('datetime64[ns]')

    # Ensure all columns have data types that xarray can handle
    dataset = dataset.astype({
        'LATITUDE': 'float64',
        'LONGITUDE': 'float64',
        'REF_TIME': 'datetime64[ns]',
        'VALUE': 'float64'
    })

    # Convert the pandas DataFrame to an xarray Dataset
    ds = dataset.set_index(['LATITUDE', 'LONGITUDE', 'REF_TIME']).to_xarray()

    # Save the dataset to a netCDF file
    ds.to_netcdf(output_file)